In [53]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

import nltk
from nltk.tokenize import word_tokenize



In [ ]:
!pip install gensim

from gensim.models.fasttext import load_facebook_vectors
from gensim.models import KeyedVectors

In [1]:
df_test = pd.read_csv('week_4_test_set.csv')

df_train = pd.read_csv('week_4_train_set.csv')

df_val = pd.read_csv('week_4_validation_set.csv')

In [3]:
df_train

,Song,Artist,Popularity,BPM,Dance,Energy,Acoustic,Happy,Loud,Camelot,Genre,Subgenre,Clean_Lyrics
0,Stretch You Out (feat. A Boogie wit da Hoodie),"Summer Walker,A Boogie Wit da Hoodie",0.5000,0.317647,0.623529,0.494737,0.252525,0.231579,0.76,12A,soul,soul rnb,get london da track niggas insecure claim enou...
1,On Melancholy Hill,Gorillaz,0.8000,0.417647,0.682353,0.726316,0.000000,0.578947,0.80,10B,electronic,electronic,melancholy hill good side consumerism like muc...
2,WHO CARES?,Rex Orange County,0.3500,0.217647,0.835294,0.242105,0.595960,0.589474,0.80,4B,indie,indie pop,mmmm mmmm mmmm mmmm first time try free doubt ...
3,Solid,Ashford & Simpson,0.4500,0.305882,0.823529,0.442105,0.272727,0.978947,0.56,8B,soul,soul 80s,love sake mistake oh forgave soon learn trust ...
4,BREAK MY SOUL,Beyoncé,0.5500,0.388235,0.682353,0.884211,0.060606,0.863158,0.84,12A,dance,dance pop,'bout explode take load bend bust open ya make...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2109,Ocean Man,Ween,0.5875,0.435294,0.717647,0.905263,0.555556,0.989474,0.80,12B,alternative,alternative rock,ocean man take hand lead land understand ocean...
2110,Paranoid Android,Radiohead,0.6750,0.676471,0.164706,0.842105,0.040404,0.189474,0.76,7B,alternative,alternative rock,please could stop noise try get rest unborn ch...
2111,"Dance, Dance",Fall Out Boy,0.7500,0.382353,0.600000,0.957895,0.010101,0.442105,0.92,10A,rock,rock,say good word bad barely stutter joke romantic...
2112,Not About Love,Fiona Apple,0.3750,0.523529,0.364706,0.389474,0.171717,0.084211,0.68,8B,alternative,alternative,early car already draw deep breath past door l...


Create a list of variables that we want to one-hot encode

In [6]:
categorical_cols = ['Camelot', 'Genre', 'Subgenre']

Fit the encoder to the train data first to prevent data leakage

In [9]:
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoder.fit(df_train[categorical_cols])

OneHotEncoder(handle_unknown='ignore', sparse_output=False)

In [11]:
# Helper function to transform and reattach
def encode_and_concat(df, encoder, categorical_cols):
    encoded = encoder.transform(df[categorical_cols])
    encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(categorical_cols), index=df.index)
    return pd.concat([df.drop(categorical_cols, axis=1), encoded_df], axis=1)

In [13]:
# Apply to each split
df_train_encoded = encode_and_concat(df_train, encoder, categorical_cols)
df_val_encoded = encode_and_concat(df_val, encoder, categorical_cols)
df_test_encoded = encode_and_concat(df_test, encoder, categorical_cols)

In [15]:
df_val_encoded

,Song,Artist,Popularity,BPM,Dance,Energy,Acoustic,Happy,Loud,Clean_Lyrics,...,Subgenre_rock 70s,Subgenre_rock 80s,Subgenre_rock 90s,Subgenre_rock alternative,Subgenre_soul,Subgenre_soul 60s,Subgenre_soul 70s,Subgenre_soul 80s,Subgenre_soul disco,Subgenre_soul rnb
0,Rebel Yell,Billy Idol,0.7375,0.688235,0.494118,0.852632,0.000000,0.484211,0.84,last night little dancer acame dancin ' door l...,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Lonely People,America,0.5750,0.176471,0.529412,0.442105,0.262626,0.557895,0.68,lonely people thinkin ' life pass give drink s...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,What A Feeling,Irene Cara,0.6375,0.429412,0.494118,0.715789,0.383838,0.600000,0.76,first nothing slow glow dream fear seem hide d...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Bird on the Wire,Leonard Cohen,0.4000,0.464706,0.341176,0.063158,0.848485,0.168421,0.40,like bird wire like drunk midnight choir try w...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,I'll Say Forever My Love,Jimmy Ruffin,0.1750,0.323529,0.517647,0.600000,0.010101,0.663158,0.76,forever forever forever love say forever love ...,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
448,True Love Will Find You in the End,Daniel Johnston,0.4250,0.411765,0.517647,0.126316,0.959596,0.115789,0.56,true love find end find friend sad know give t...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
449,Dem Boyz,Boyz N Da Hood,0.4500,0.176471,0.788235,0.610526,0.000000,0.284211,0.80,bad boy south block entertainment see block bo...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
450,Come A Little Bit Closer,Jay & The Americans,0.5500,0.494118,0.623529,0.663158,0.525253,0.894737,0.68,little caf side border uh sit givin ' look mak...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
451,Space Age Love Song,A Flock Of Seagulls,0.4250,0.535294,0.447059,0.494737,0.000000,0.621053,0.52,see eye make smile little fall love see eye to...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [17]:
list(df_val_encoded.columns)

['Song',
 'Artist',
 'Popularity',
 'BPM',
 'Dance',
 'Energy',
 'Acoustic',
 'Happy',
 'Loud',
 'Clean_Lyrics',
 'Camelot_10A',
 'Camelot_10B',
 'Camelot_11A',
 'Camelot_11B',
 'Camelot_12A',
 'Camelot_12B',
 'Camelot_1A',
 'Camelot_1B',
 'Camelot_2A',
 'Camelot_2B',
 'Camelot_3A',
 'Camelot_3B',
 'Camelot_4A',
 'Camelot_4B',
 'Camelot_5A',
 'Camelot_5B',
 'Camelot_6A',
 'Camelot_6B',
 'Camelot_7A',
 'Camelot_7B',
 'Camelot_8A',
 'Camelot_8B',
 'Camelot_9A',
 'Camelot_9B',
 'Genre_alternative',
 'Genre_country',
 'Genre_dance',
 'Genre_electronic',
 'Genre_folk',
 'Genre_hip hop',
 'Genre_indie',
 'Genre_pop',
 'Genre_rnb',
 'Genre_rock',
 'Genre_soul',
 'Subgenre_alternative',
 'Subgenre_alternative rnb',
 'Subgenre_alternative rock',
 'Subgenre_country',
 'Subgenre_country pop',
 'Subgenre_dance',
 'Subgenre_dance electronic',
 'Subgenre_dance pop',
 'Subgenre_electronic',
 'Subgenre_electronic dance',
 'Subgenre_electronic pop',
 'Subgenre_folk',
 'Subgenre_folk rock',
 'Subgenre_h

In [19]:
df_train_encoded.head()

,Song,Artist,Popularity,BPM,Dance,Energy,Acoustic,Happy,Loud,Clean_Lyrics,...,Subgenre_rock 70s,Subgenre_rock 80s,Subgenre_rock 90s,Subgenre_rock alternative,Subgenre_soul,Subgenre_soul 60s,Subgenre_soul 70s,Subgenre_soul 80s,Subgenre_soul disco,Subgenre_soul rnb
0,Stretch You Out (feat. A Boogie wit da Hoodie),"Summer Walker,A Boogie Wit da Hoodie",0.50,0.317647,0.623529,0.494737,0.252525,0.231579,0.76,get london da track niggas insecure claim enou...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,On Melancholy Hill,Gorillaz,0.80,0.417647,0.682353,0.726316,0.000000,0.578947,0.80,melancholy hill good side consumerism like muc...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,WHO CARES?,Rex Orange County,0.35,0.217647,0.835294,0.242105,0.595960,0.589474,0.80,mmmm mmmm mmmm mmmm first time try free doubt ...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Solid,Ashford & Simpson,0.45,0.305882,0.823529,0.442105,0.272727,0.978947,0.56,love sake mistake oh forgave soon learn trust ...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,BREAK MY SOUL,Beyoncé,0.55,0.388235,0.682353,0.884211,0.060606,0.863158,0.84,'bout explode take load bend bust open ya make...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [21]:
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/langleyburke/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [22]:
def simple_tokenize(text):
  return text.split()

In [25]:
# Apply to all three DataFrames
for df in [df_train_encoded, df_val_encoded, df_test_encoded]:
    df['Tokens'] = df['Clean_Lyrics'].apply(simple_tokenize)

In [27]:
df_train_encoded

,Song,Artist,Popularity,BPM,Dance,Energy,Acoustic,Happy,Loud,Clean_Lyrics,...,Subgenre_rock 80s,Subgenre_rock 90s,Subgenre_rock alternative,Subgenre_soul,Subgenre_soul 60s,Subgenre_soul 70s,Subgenre_soul 80s,Subgenre_soul disco,Subgenre_soul rnb,Tokens
0,Stretch You Out (feat. A Boogie wit da Hoodie),"Summer Walker,A Boogie Wit da Hoodie",0.5000,0.317647,0.623529,0.494737,0.252525,0.231579,0.76,get london da track niggas insecure claim enou...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,"[get, london, da, track, niggas, insecure, cla..."
1,On Melancholy Hill,Gorillaz,0.8000,0.417647,0.682353,0.726316,0.000000,0.578947,0.80,melancholy hill good side consumerism like muc...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"[melancholy, hill, good, side, consumerism, li..."
2,WHO CARES?,Rex Orange County,0.3500,0.217647,0.835294,0.242105,0.595960,0.589474,0.80,mmmm mmmm mmmm mmmm first time try free doubt ...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"[mmmm, mmmm, mmmm, mmmm, first, time, try, fre..."
3,Solid,Ashford & Simpson,0.4500,0.305882,0.823529,0.442105,0.272727,0.978947,0.56,love sake mistake oh forgave soon learn trust ...,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,"[love, sake, mistake, oh, forgave, soon, learn..."
4,BREAK MY SOUL,Beyoncé,0.5500,0.388235,0.682353,0.884211,0.060606,0.863158,0.84,'bout explode take load bend bust open ya make...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"['bout, explode, take, load, bend, bust, open,..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2109,Ocean Man,Ween,0.5875,0.435294,0.717647,0.905263,0.555556,0.989474,0.80,ocean man take hand lead land understand ocean...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"[ocean, man, take, hand, lead, land, understan..."
2110,Paranoid Android,Radiohead,0.6750,0.676471,0.164706,0.842105,0.040404,0.189474,0.76,please could stop noise try get rest unborn ch...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"[please, could, stop, noise, try, get, rest, u..."
2111,"Dance, Dance",Fall Out Boy,0.7500,0.382353,0.600000,0.957895,0.010101,0.442105,0.92,say good word bad barely stutter joke romantic...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"[say, good, word, bad, barely, stutter, joke, ..."
2112,Not About Love,Fiona Apple,0.3750,0.523529,0.364706,0.389474,0.171717,0.084211,0.68,early car already draw deep breath past door l...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"[early, car, already, draw, deep, breath, past..."


In [29]:
num_songs = len(df_train_encoded)

print(num_songs)

2114


Because the number of songs is below 10000 we will use the FastText to vectorize the tokens. Download it here before running the code :  https://fasttext.cc/docs/en/english-vectors.html. "English" - crawl-300d-2M-subword.zip

In [39]:
#Change path to your own when running so no errors occur - the file can be found in the data folder for now
glove_path = '/Users/langleyburke/Downloads/crawl-300d-2M-subword/crawl-300d-2M-subword.vec'

fasttext_model = KeyedVectors.load_word2vec_format(glove_path, binary=False)


Testing to see if it is all set up properly.

In [41]:
fasttext_model['love']

array([ 2.830e-02, -6.750e-02,  1.439e-01,  4.900e-02, -5.410e-02,
       -6.450e-02,  1.784e-01,  1.900e-02,  1.581e-01, -5.000e-04,
       -6.250e-02, -8.160e-02,  5.070e-02,  1.950e-02, -2.430e-02,
        9.000e-03, -1.690e-02,  1.800e-02, -2.660e-02,  7.450e-02,
       -4.700e-03,  2.460e-02, -7.040e-02, -2.320e-02,  1.196e-01,
        2.390e-02,  3.600e-02, -1.230e-02,  8.310e-02,  1.073e-01,
       -4.320e-02, -1.167e-01, -2.840e-02, -1.630e-02, -1.980e-02,
        4.480e-02,  8.810e-02,  1.280e-02, -9.140e-02,  3.460e-02,
       -1.268e-01, -1.388e-01, -1.340e-02,  8.290e-02, -2.490e-02,
        1.423e-01, -7.780e-02, -3.240e-02, -1.005e-01,  1.460e-02,
        3.700e-03, -1.240e-02,  1.042e-01,  9.220e-02, -8.640e-02,
        8.580e-02, -1.330e-02, -5.090e-02, -1.786e-01,  5.330e-02,
        5.830e-02,  1.780e-02, -6.600e-03,  1.740e-02, -1.740e-02,
        4.500e-02, -1.700e-02,  2.280e-02, -6.700e-03, -4.960e-02,
        2.500e-02, -2.030e-02, -2.480e-02, -7.900e-02,  3.830e

Seems like it is working, will now turn the tokenized lyrics into vectors so we can use them in our recommender model! 

In [47]:
def get_fasttext_embedding(tokens, model, dim=300):
    vectors = [model[word] for word in tokens if word in model]
    if not vectors:
        return np.zeros(dim)
    return np.mean(vectors, axis=0)

In [55]:
#Creating new column 'Embeddings' for the final lyric conversions

for df in [df_train_encoded, df_val_encoded, df_test_encoded]:
    df['Embeddings'] = df['Tokens'].apply(lambda tokens: get_fasttext_embedding(tokens, fasttext_model))


In [57]:
df_train_encoded.head()

,Song,Artist,Popularity,BPM,Dance,Energy,Acoustic,Happy,Loud,Clean_Lyrics,...,Subgenre_rock 90s,Subgenre_rock alternative,Subgenre_soul,Subgenre_soul 60s,Subgenre_soul 70s,Subgenre_soul 80s,Subgenre_soul disco,Subgenre_soul rnb,Tokens,Embeddings
0,Stretch You Out (feat. A Boogie wit da Hoodie),"Summer Walker,A Boogie Wit da Hoodie",0.50,0.317647,0.623529,0.494737,0.252525,0.231579,0.76,get london da track niggas insecure claim enou...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,"[get, london, da, track, niggas, insecure, cla...","[-0.054992, -0.04241732, 0.12042137, -0.006478..."
1,On Melancholy Hill,Gorillaz,0.80,0.417647,0.682353,0.726316,0.000000,0.578947,0.80,melancholy hill good side consumerism like muc...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"[melancholy, hill, good, side, consumerism, li...","[-0.014098572, -0.04454142, 0.11146287, 0.0116..."
2,WHO CARES?,Rex Orange County,0.35,0.217647,0.835294,0.242105,0.595960,0.589474,0.80,mmmm mmmm mmmm mmmm first time try free doubt ...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"[mmmm, mmmm, mmmm, mmmm, first, time, try, fre...","[-0.0018581813, -0.036476366, 0.14796545, 0.00..."
3,Solid,Ashford & Simpson,0.45,0.305882,0.823529,0.442105,0.272727,0.978947,0.56,love sake mistake oh forgave soon learn trust ...,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,"[love, sake, mistake, oh, forgave, soon, learn...","[-0.013666291, -0.011470232, 0.11901681, 0.021..."
4,BREAK MY SOUL,Beyoncé,0.55,0.388235,0.682353,0.884211,0.060606,0.863158,0.84,'bout explode take load bend bust open ya make...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"['bout, explode, take, load, bend, bust, open,...","[-0.056397956, -0.050341, 0.10335694, -0.00435..."


In [ ]:
We can see now the tokens are in vector form and ready for use.